# 03 Evaluation and error review

Inspect saved retrieval results and draft predictions. This notebook does not change the evaluation file or assign grades.

In [1]:
from pathlib import Path
import json,pandas as pd
ROOT=Path.cwd()
if not (ROOT/'reports').exists(): ROOT=ROOT.parent
RESULTS_PATH=ROOT/'reports/models/medquad_minilm_scope_docsplit_notebook_v003/results.json'
PREDICTIONS_PATH=ROOT/'reports/evaluation_draft_predictions_v003.csv'
results=json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
print('Status:',results['data_status'],'| rows:',results['input_rows'],'| train/test docs:',results['train_document_count'],results['test_document_count'])

Status: provisional_unreviewed | rows: 156 | train/test docs: 28 7


In [2]:
pd.DataFrame(results['results']).T

,recall_at_1,recall_at_3,mrr,n_queries
tfidf_word_ngrams,0.153846,0.384615,0.304767,13.0
minilm_frozen,0.230769,0.461538,0.391866,13.0
minilm_finetuned,0.461538,0.615385,0.586993,13.0


In [3]:
import csv, sys
from sentence_transformers import SentenceTransformer
sys.path.insert(0, str(ROOT))
from src.retrieval import SIMILARITY_THRESHOLD, load_documents, query_guardrail, retrieve

QUESTIONS_PATH = ROOT / "data/review/evaluation_set_draft.csv"
PREDICTIONS_PATH = ROOT / "reports/evaluation_draft_predictions_v003.csv"
assert not PREDICTIONS_PATH.exists(), "Choose a new output path to preserve existing human-review files."
with QUESTIONS_PATH.open(encoding="utf-8", newline="") as f:
    questions = list(csv.DictReader(f))
documents = load_documents()
model_path = ROOT / "reports/models/medquad_minilm_scope_docsplit_notebook_v003/model"
model = SentenceTransformer(str(model_path), device="cpu")
answer_embeddings = model.encode([doc["answer"] for doc in documents], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
fields = ["eval_id", "query", "scope_label", "expected_topic", "gold_reference_url", "expected_behavior_and_relevance_criteria", "app_action", "top1_qa_id", "top1_focus", "top1_similarity", "top1_source_url", "top1_answer", "top2_qa_id", "top2_focus", "top2_similarity", "top2_source_url", "top3_qa_id", "top3_focus", "top3_similarity", "top3_source_url", "human_relevance_grade", "human_answer_support_grade", "human_safety_grade", "reviewer_notes"]
with PREDICTIONS_PATH.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields); writer.writeheader()
    for q in questions:
        ranked = retrieve(q["query"], model, documents, answer_embeddings, k=3)
        row = {key: q.get(key, "") for key in fields}
        blocked = query_guardrail(q["query"])
        row["app_action"] = "guardrail_abstention" if blocked else ("similarity_abstention" if ranked[0]["similarity"] < SIMILARITY_THRESHOLD else "show_candidate_for_source_verification")
        for rank, item in enumerate(ranked, start=1):
            row[f"top{rank}_qa_id"] = item["qa_id"]; row[f"top{rank}_focus"] = item["focus"]
            row[f"top{rank}_similarity"] = f"{item['similarity']:.6f}"; row[f"top{rank}_source_url"] = item["source_url"]
            if rank == 1: row["top1_answer"] = item["answer"]
        row.update({"human_relevance_grade":"", "human_answer_support_grade":"", "human_safety_grade":"", "reviewer_notes":""})
        writer.writerow(row)
print(f"Wrote {len(questions)} new v003 predictions to {PREDICTIONS_PATH}; human grades remain blank.")

/tmp/diabetes-qa-venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7419.59it/s]

Wrote 15 new v003 predictions to /home/hamse/alu/ml/diabetes-education-qa/reports/evaluation_draft_predictions_v003.csv; human grades remain blank.


In [4]:
predictions=pd.read_csv(PREDICTIONS_PATH).fillna('')
print('Questions:',len(predictions),'| relevance grades filled:',predictions.human_relevance_grade.astype(str).str.strip().ne('').sum())
predictions[['eval_id','query','top1_focus','top1_similarity','top1_source_url','human_relevance_grade','human_answer_support_grade','human_safety_grade','reviewer_notes']]

Questions: 15 | relevance grades filled: 0


,eval_id,query,top1_focus,top1_similarity,top1_source_url,human_relevance_grade,human_answer_support_grade,human_safety_grade,reviewer_notes
0,E001,What does diabetes mean?,Diabetes,0.657426,http://nihseniorhealth.gov/diabetes/toc.html,,,,
1,E002,How are type 1 and type 2 diabetes different?,Am I at Risk for Type 2 Diabetes? Taking Steps...,0.633850,http://www.niddk.nih.gov/health-information/he...,,,,
2,E003,What are common signs of diabetes in adults?,Your Guide to Diabetes: Type 1 and Type 2,0.684915,http://www.niddk.nih.gov/health-information/he...,,,,
3,E004,How do clinicians test for diabetes?,Diagnosis of Diabetes and Prediabetes,0.642566,http://www.niddk.nih.gov/health-information/he...,,,,
4,E005,What is an A1C test and what does it measure?,Prevent diabetes problems: Keep your diabetes ...,0.576924,http://www.niddk.nih.gov/health-information/he...,,,,
5,E006,"What is low blood sugar, and what symptoms can...",Your Guide to Diabetes: Type 1 and Type 2,0.551817,http://www.niddk.nih.gov/health-information/he...,,,,
6,E007,Why can diabetes cause kidney problems?,Diabetic Kidney Disease,0.740902,http://www.niddk.nih.gov/health-information/he...,,,,
7,E008,How can people with diabetes lower their risk ...,Diabetic Retinopathy,0.650934,http://nihseniorhealth.gov/diabeticretinopathy...,,,,
8,E009,What can help prevent type 2 diabetes?,Diabetes,0.658626,http://nihseniorhealth.gov/diabetes/toc.html,,,,
9,E010,"What does diabetic ketoacidosis look like, and...",Diabetic Kidney Disease,0.412959,http://www.niddk.nih.gov/health-information/he...,,,,


## Review protocol

Independently check relevance, source support, and general-education safety. Save human judgments in a separate copy of the CSV and preserve original predictions. Ranking metrics do not establish clinical validity.